# 10 — 链接预测、负采样与图自编码器

前两课分别预测节点标签和整张图的标签；这一课预测一对节点之间是否存在边。我们将在 Cora 上隐藏一部分真实引用边，让模型根据剩余图结构和论文特征把它们找回来。

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch_geometric.utils import negative_sampling

ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from crosscity.data.citation import load_planetoid
from crosscity.data.link_prediction import make_link_prediction_splits
from crosscity.models.link_prediction import build_gae, build_vgae
from crosscity.training.link_prediction import feature_similarity_baseline, train_link_predictor
from crosscity.utils import seed_everything
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

## 1. 什么是负采样？

图数据明确列出了存在的边，它们是正样本。例如 `(0, 1)` 是引用关系。但训练二分类器还需要反例：哪些节点对不应该连边？一个 N 节点无向图最多有 $N(N-1)/2$ 个不含自环的节点对，真实边通常只占极小部分。把全部非边都用于训练既昂贵又会造成极端类别不平衡，因此从非边集合中随机抽出一部分作为负样本，这就是 negative sampling。

负样本的含义是“当前数据中没有观察到这条边”，不一定是现实世界中绝对不可能存在。推荐系统和知识图谱里尤其要注意这种区别。

In [ ]:
# 五个节点中，先定义四条无向正边；edge_index 用两个方向保存。
positive_undirected = [(0, 1), (1, 2), (2, 3), (3, 4)]
positive_directed = positive_undirected + [(j, i) for i, j in positive_undirected]
tiny_edge_index = torch.tensor(positive_directed).T
all_pairs = [(i, j) for i in range(5) for j in range(i + 1, 5)]
non_edges = [pair for pair in all_pairs if pair not in positive_undirected]
sampled_negative = negative_sampling(tiny_edge_index, num_nodes=5,
                                     num_neg_samples=len(positive_undirected),
                                     force_undirected=True)
print('全部可能节点对:', all_pairs)
print('正边:', positive_undirected)
print('候选非边:', non_edges)
print('本轮抽到的负边:', sampled_negative.T.tolist())

## 2. 链接预测切分为什么容易泄漏？

不能先让 GNN 在完整图上传播消息，再声称预测被模型看过的边。正确流程是先隐藏 validation/test 正边，再只用训练边构造 encoder 的消息图。`RandomLinkSplit` 同时生成：

- `edge_index`：encoder 真正允许看到的消息边；
- `pos_edge_label_index`：需要判断为存在的正边；
- `neg_edge_label_index`：从原始完整图的非边中抽出的负边。

本项目在切分阶段生成训练负边，保证被隐藏的 validation/test 正边不会被误采成训练负边。

In [ ]:
data = load_planetoid(ROOT / 'data/raw/planetoid', 'cora')
splits = make_link_prediction_splits(data, seed=42)
for name, graph in [('train', splits.train), ('validation', splits.validation), ('test', splits.test)]:
    print(name, {'message edge entries': graph.edge_index.shape[1],
                 'positive labels': graph.pos_edge_label_index.shape[1],
                 'negative labels': graph.neg_edge_label_index.shape[1]})
assert splits.train.edge_index.shape[1] < data.edge_index.shape[1]

## 3. GAE：把图压缩成节点嵌入，再重建边

GCN encoder 生成每个节点的潜在向量：

$$Z = \operatorname{GCNEncoder}(X, A_{train}).$$

内积 decoder 判断两个节点是否适合连接：

$$p((i,j)\in E)=\sigma(z_i^Tz_j).$$

训练目标让正边分数升高、负边分数降低。它叫 Graph Autoencoder，是因为输入图经过 encoder 压缩为 Z，再由 decoder 尝试重建邻接关系。

## 4. VGAE：节点不再是一个点，而是一个分布

GAE 为节点学习确定向量 $z_i$；VGAE 为节点学习高斯分布：

$$q(z_i|X,A)=\mathcal N(\mu_i,\operatorname{diag}(\sigma_i^2)).$$

通过 $z_i=\mu_i+\sigma_i\odot\epsilon$ 采样，并加入 KL loss，让潜在空间接近标准正态分布。VGAE 因而可以表达不确定性并生成平滑潜在空间，但正则化也可能让简单任务的重建精度下降。

## 5. 为什么不用普通 accuracy？

真实大图的非边远多于正边。如果把所有非边都算进去，一个永远预测“没有边”的模型也可能得到接近 100% accuracy。

- ROC-AUC：随机取一个正边和一个负边，模型把正边排在前面的概率；
- Average Precision：查看按分数从高到低检索边时，precision-recall 曲线的整体质量，对正负不平衡更敏感。

两者都是越接近 1 越好，随机排序约为 0.5（平衡评估集下 AP 也约为 0.5）。

In [ ]:
baseline = feature_similarity_baseline(splits.test)
print('feature cosine baseline:', baseline)
for name, builder in [('GAE', build_gae), ('VGAE', build_vgae)]:
    seed_everything(42)
    result = train_link_predictor(builder(data.num_features, 32, 16),
                                  splits.train, splits.validation, splits.test,
                                  device=device, max_epochs=5, patience=5)
    print(name, 'val AP=', round(result.validation.average_precision, 3),
          'test AUC=', round(result.test.auc, 3))

## 6. 三种子正式实验

固定同一个 link split，只改变模型初始化。候选模型、早停指标和 test 集已经预先确定：按照 validation AP 恢复最佳 checkpoint，最后报告 test AUC/AP。

In [ ]:
records, histories = [], {}
for seed in [42, 43, 44]:
    for name, builder in [('GAE', build_gae), ('VGAE', build_vgae)]:
        seed_everything(seed)
        result = train_link_predictor(builder(data.num_features, 64, 32),
                                      splits.train, splits.validation, splits.test,
                                      device=device, max_epochs=300, patience=50)
        records.append({'seed': seed, 'model': name, 'best_epoch': result.best_epoch,
                        'validation_auc': result.validation.auc,
                        'validation_ap': result.validation.average_precision,
                        'test_auc': result.test.auc,
                        'test_ap': result.test.average_precision})
        histories[(seed, name)] = result.history
results = pd.DataFrame(records)
display(results.sort_values(['seed', 'validation_ap'], ascending=[True, False]))
results.groupby('model')[['validation_auc', 'validation_ap', 'test_auc', 'test_ap']].agg(['mean', 'std']).style.format('{:.3f}')

In [ ]:
for (seed, name), history in histories.items():
    if seed == 42:
        frame = pd.DataFrame(history)
        plt.plot(frame.epoch, frame.val_ap, label=name)
plt.xlabel('epoch'); plt.ylabel('validation AP'); plt.legend(); plt.show()

## 7. 解释清单

1. GAE/VGAE 是否超过只比较节点特征余弦相似度的基线？
2. AUC 高但 AP 较低意味着什么？
3. VGAE 的 KL 正则化带来了更稳定的结果，还是造成欠拟合？
4. 如果负采样比例从 1:1 改成 1:10，accuracy、AUC 和 AP 会怎样变化？
5. Cora 的未观察边是否一定是真负边？这对评价有什么影响？